In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [4]:
pip install pyreadstat

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.7/2.7 MB 86.6 MB/s eta 0:00:00


In [1]:
import pyreadstat
import pandas as pd

file_path = '/content/drive/My Drive/IAIR7EFL.DTA'  # Adjust path if inside a subfolder
target_cols = ['v012', 'v024', 'v025', 'v190', 'v213', 'v215']

filtered_chunks = []

# Stream chunks via C-engine parser
reader = pyreadstat.read_file_in_chunks(
    pyreadstat.read_dta,
    file_path,
    chunksize=100000,
    usecols=target_cols,
    apply_value_formats=False,        # Disables slow string label parsing
    disable_datetime_conversion=True  # Bypasses date parsing overhead
)

print("Processing NFHS file in chunks...")
for chunk, meta in reader:
    # Filter age 30 to 55 per chunk
    subset = chunk[(chunk['v012'] >= 30) & (chunk['v012'] <= 55)]
    filtered_chunks.append(subset)

# Merge filtered chunks
df_nfhs_filtered = pd.concat(filtered_chunks, ignore_index=True)
print(f"Extraction complete! Filtered rows: {len(df_nfhs_filtered)}")

# Save lightweight file (~10 MB) to Drive
df_nfhs_filtered.to_parquet('/content/drive/My Drive/nfhs_menopause_subset.parquet')
print("Saved lightweight parquet file to Google Drive.")

Processing NFHS file in chunks...
Extraction complete! Filtered rows: 364556
Saved lightweight parquet file to Google Drive.


In [2]:
df_nfhs_filtered = pd.read_parquet('/content/drive/My Drive/nfhs_menopause_subset.parquet')

In [3]:
import pandas as pd
import numpy as np

# Load Parquet file instantly
parquet_path = '/content/drive/My Drive/nfhs_menopause_subset.parquet'
df_nfhs = pd.read_parquet(parquet_path)

# Map & clean NFHS features
df_nfhs_clean = pd.DataFrame()
df_nfhs_clean['age'] = df_nfhs['v012'].astype(int)
df_nfhs_clean['residence_type'] = df_nfhs['v025']  # Urban / Rural
df_nfhs_clean['wealth_index'] = df_nfhs['v190']    # Wealth quintile (1-5)
df_nfhs_clean['is_pregnant'] = (df_nfhs['v213'] == 1).astype(int)
df_nfhs_clean['source'] = 'nfhs'

# Derive menopause/perimenopause stage from v215 (time since last menstrual period)
def classify_nfhs_stage(val):
    val_str = str(val).lower()
    if 'menopause' in val_str or 'no period' in val_str or '956' in val_str:
        return 'Postmenopausal'
    elif '0' in val_str or 'days' in val_str or 'weeks' in val_str or 'regular' in val_str:
        return 'Premenopausal / Perimenopausal'
    return 'Perimenopausal'

df_nfhs_clean['menopause_stage'] = df_nfhs['v215'].apply(classify_nfhs_stage)

print("NFHS Cleaned Dataset Shape:", df_nfhs_clean.shape)
print(df_nfhs_clean['menopause_stage'].value_counts())

NFHS Cleaned Dataset Shape: (364556, 6)
menopause_stage
Premenopausal / Perimenopausal    312910
Perimenopausal                     51646
Name: count, dtype: int64


In [4]:
# Load survey CSV
df_survey = pd.read_csv('Menopause & Perimenopause Symptoms Survey .csv')

# Map age ranges to numerical midpoints
age_map = {'Under 40': 38, '40-44': 42, '45-49': 47, '50-54': 52, '55-59': 57}
freq_map = {'Never': 0, 'Rarely (1–2 times a month)': 1, 'Occasionally (1–2 times a week)': 2, 'Frequently (Daily, 1–3 times a day)': 3}

df_survey_clean = pd.DataFrame()
df_survey_clean['age'] = df_survey['What is your current age range?'].map(age_map)
df_survey_clean['hot_flashes_severity'] = df_survey['How often do you experience hot flashes or sudden flushes of heat?'].map(freq_map).fillna(0)

# Classify survey stage
def classify_survey_stage(status):
    status_str = str(status).lower()
    if '12 consecutive months' in status_str or 'surgical' in status_str:
        return 'Postmenopausal'
    elif 'irregular' in status_str:
        return 'Perimenopausal'
    return 'Premenopausal / Perimenopausal'

df_survey_clean['menopause_stage'] = df_survey['Which statement best describes your current menstrual status?'].apply(classify_survey_stage)
df_survey_clean['source'] = 'survey'

print("Survey Cleaned Dataset Shape:", df_survey_clean.shape)

Survey Cleaned Dataset Shape: (28, 4)


In [5]:
# Save cleaned outputs back to Google Drive
df_nfhs_clean.to_csv('/content/drive/My Drive/clean_nfhs_menopause.csv', index=False)
df_survey_clean.to_csv('/content/drive/My Drive/clean_survey_symptoms.csv', index=False)

print("Data preparation complete! Files ready for modeling.")

Data preparation complete! Files ready for modeling.


In [6]:
import pandas as pd

# --- 1. INSPECT NFHS PARQUET FILE ---
parquet_path = '/content/drive/My Drive/nfhs_menopause_subset.parquet'
df_nfhs = pd.read_parquet(parquet_path)

print("="*50)
print("NFHS DATASET SUMMARY")
print("="*50)
print(f"Shape: {df_nfhs.shape[0]} rows, {df_nfhs.shape[1]} columns\n")
print("Columns & Data Types:")
print(df_nfhs.dtypes)
print("\nNull Value Counts:")
print(df_nfhs.isnull().sum())
print("\nFirst 3 Rows:")
print(df_nfhs.head(3))


# --- 2. INSPECT GOOGLE FORM SURVEY DATASET ---
df_survey = pd.read_csv('Menopause & Perimenopause Symptoms Survey .csv')

print("\n" + "="*50)
print("GOOGLE FORM SURVEY SUMMARY")
print("="*50)
print(f"Shape: {df_survey.shape[0]} rows, {df_survey.shape[1]} columns\n")
print("Null Value Counts per Question:")
print(df_survey.isnull().sum())

NFHS DATASET SUMMARY
Shape: 364556 rows, 6 columns

Columns & Data Types:
v012    int64
v024    int64
v025    int64
v190    int64
v213    int64
v215    int64
dtype: object

Null Value Counts:
v012    0
v024    0
v025    0
v190    0
v213    0
v215    0
dtype: int64

First 3 Rows:
   v012  v024  v025  v190  v213  v215
0    38     1     2     3     0   301
1    40     1     2     3     0   201
2    38     1     2     1     0   993

GOOGLE FORM SURVEY SUMMARY
Shape: 28 rows, 23 columns

Null Value Counts per Question:
Timestamp                                                                                                                            0
What is your current age range?                                                                                                      0
Which statement best describes your current menstrual status?                                                                        0
How often do you experience hot flashes or sudden flushes of heat?         

In [8]:
import pandas as pd

# Load cleaned survey data
df_survey = pd.read_csv('/content/drive/My Drive/clean_survey_symptoms.csv')

print("=== SURVEY DATASET SUMMARY ===")
print("Total Responses:", len(df_survey))
print("\nMenopause Stage Breakdown:")
print(df_survey['menopause_stage'].value_counts(normalize=True) * 100)

# Load cleaned NFHS data
df_nfhs = pd.read_parquet('/content/drive/My Drive/nfhs_menopause_subset.parquet')

print("\n=== NFHS DATASET SUMMARY ===")
print("Total Records:", len(df_nfhs))
print("Age Mean:", df_nfhs['v012'].mean().round(2))
print("Age Range:", df_nfhs['v012'].min(), "-", df_nfhs['v012'].max())

=== SURVEY DATASET SUMMARY ===
Total Responses: 28

Menopause Stage Breakdown:
menopause_stage
Premenopausal / Perimenopausal    50.0
Perimenopausal                    25.0
Postmenopausal                    25.0
Name: proportion, dtype: float64

=== NFHS DATASET SUMMARY ===
Total Records: 364556
Age Mean: 38.8
Age Range: 30 - 49
